**Status: Race results ingestion (2018 to today).** Light-weight companion to `phase_1.ipynb`: one row per driver per race with grid, finishing position, team, status and qualifying gap to pole. No laps or telemetry.

Primary source is the Ergast-compatible results API (one request per season), which avoids FastF1's 500-calls-per-hour live-timing limit. Races the Ergast mirror has not published yet (it can lag by several rounds) are filled in from FastF1 when its rate limit allows; anything skipped is picked up on the next run.

Writes `workspace.default.f1_race_results`. Re-run after every race weekend, then re-run `phase_3_podium.ipynb`.

In [ ]:
import os
import re
import tempfile
import time

import fastf1
import numpy as np
import pandas as pd
from fastf1.ergast import Ergast

RESULTS_TABLE = "workspace.default.f1_race_results"
FIRST_YEAR = 2018

cache_dir = os.path.join(tempfile.gettempdir(), "fastf1_cache")
os.makedirs(cache_dir, exist_ok=True)
fastf1.Cache.enable_cache(cache_dir)
erg = Ergast()

In [ ]:
# lineage so a rebranded team keeps its history (Toro Rosso -> AlphaTauri -> RB -> Racing Bulls, Sauber -> Alfa Romeo -> Audi, etc.)
def team_key(name):
    n = str(name).lower()
    for needle, key in [
        ("toro rosso", "Racing Bulls"), ("alphatauri", "Racing Bulls"), ("rb f1", "Racing Bulls"), ("racing bulls", "Racing Bulls"), ("visa", "Racing Bulls"),
        ("renault", "Alpine"), ("alpine", "Alpine"),
        ("force india", "Aston Martin"), ("racing point", "Aston Martin"), ("aston martin", "Aston Martin"),
        ("sauber", "Audi"), ("alfa romeo", "Audi"), ("audi", "Audi"),
        ("red bull", "Red Bull Racing"), ("haas", "Haas F1 Team"), ("cadillac", "Cadillac"),
        ("mercedes", "Mercedes"), ("ferrari", "Ferrari"), ("mclaren", "McLaren"), ("williams", "Williams"),
    ]:
        if needle in n:
            return key
    return str(name)

def secs(s):
    return pd.to_timedelta(s).dt.total_seconds()

def polite(fn, **kw):
    # throttle and retry: the mirror answers 'Too Many Requests' on bursts
    for wait in (1, 5, 20, 60, 120):
        time.sleep(1.0)
        try:
            return fn(**kw)
        except Exception as e:
            print("   retry after", type(e).__name__, "-", wait, "s")
            time.sleep(wait)
    return fn(**kw)

def pages(fn, year, step=100, max_pages=12):
    # the mirror caps a request at 100 rows, so walk offsets and stitch a race split across two pages back together
    desc, content = {}, {}
    for k in range(max_pages):
        r = polite(fn, season=year, limit=step, offset=k * step)
        if not len(r.content):
            break
        for d, c in zip(r.description.to_dict("records"), r.content):
            rnd = int(d["round"])
            desc[rnd] = d
            content[rnd] = pd.concat([content[rnd], c]) if rnd in content else c
        if sum(len(c) for c in r.content) < step:
            break
    return desc, {k: v.drop_duplicates("driverCode") for k, v in content.items()}

def ergast_season(year):
    res_d, res_c = pages(erg.get_race_results, year)
    qual_d, qmap = pages(erg.get_qualifying_results, year)
    circ = polite(erg.get_circuits, season=year, limit=100).set_index("circuitId")["circuitName"].to_dict()
    rows = []
    for rnd in sorted(res_d):
        d, r = res_d[rnd], res_c[rnd]
        q = qmap.get(rnd)
        out = pd.DataFrame({
            "Year": year, "Round": int(d["round"]), "Race": d["raceName"], "Location": d["circuitId"], "CircuitName": circ.get(d["circuitId"], d["circuitId"]),
            "EventDate": str(d["raceDate"])[:10],
            "Driver": r["driverCode"].values, "DriverNumber": r["driverNumber"].astype(str).values,
            "FullName": (r["givenName"] + " " + r["familyName"]).values,
            "TeamName": r["constructorName"].values, "TeamKey": [team_key(t) for t in r["constructorName"]],
            "GridPosition": r["grid"].astype(float).values, "FinalPosition": pd.to_numeric(r["position"], errors="coerce").values,
            "Status": r["status"].astype(str).values, "Points": r["points"].astype(float).values,
        })
        if q is not None and len(q):
            best = pd.concat([secs(q[c]) for c in ("Q1", "Q2", "Q3")], axis=1).min(axis=1)
            qq = pd.DataFrame({"Driver": q["driverCode"].values, "QualiPosition": pd.to_numeric(q["position"], errors="coerce").values, "QualiBest": best.values})
            out = out.merge(qq, on="Driver", how="left")
            out["QualiGapPct"] = (out["QualiBest"] / out["QualiBest"].min() - 1) * 100
            out = out.drop(columns=["QualiBest"])
        else:
            out["QualiPosition"], out["QualiGapPct"] = np.nan, np.nan
        out["GridPosition"] = np.where(out["GridPosition"].fillna(0) <= 0, len(out), out["GridPosition"])
        rows.append(out)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

In [ ]:
today = pd.Timestamp.now().normalize()
frames = []
for year in range(FIRST_YEAR, today.year + 1):
    f = ergast_season(year)
    print(year, "Ergast races:", f["Round"].nunique() if len(f) else 0)
    frames.append(f)
results = pd.concat(frames, ignore_index=True)

In [ ]:
# fill 2026-style gaps: races on the calendar that Ergast has not published yet (FastF1 live timing, rate limited)
circuit_by_race = results.drop_duplicates("Race").set_index("Race")[["Location", "CircuitName"]].to_dict("index")

def fastf1_race(year, ev):
    race = fastf1.get_session(year, int(ev["RoundNumber"]), "R")
    race.load(laps=False, telemetry=False, weather=False, messages=False)
    r = race.results
    out = pd.DataFrame({
        "Year": year, "Round": int(ev["RoundNumber"]), "Race": ev["EventName"],
        "Location": circuit_by_race.get(ev["EventName"], {}).get("Location", re.sub(r"\W+", "_", str(ev["Location"]).lower())),
        "CircuitName": circuit_by_race.get(ev["EventName"], {}).get("CircuitName", ev["Location"]),
        "EventDate": pd.Timestamp(ev["EventDate"]).strftime("%Y-%m-%d"),
        "Driver": r["Abbreviation"].values, "DriverNumber": r["DriverNumber"].astype(str).values, "FullName": r["FullName"].values,
        "TeamName": r["TeamName"].values, "TeamKey": [team_key(t) for t in r["TeamName"]],
        "GridPosition": r["GridPosition"].astype(float).values, "FinalPosition": r["Position"].values, "Status": r["Status"].astype(str).values, "Points": r["Points"].values,
    })
    try:
        qs = fastf1.get_session(year, int(ev["RoundNumber"]), "Q")
        qs.load(laps=False, telemetry=False, weather=False, messages=False)
        q = qs.results
        best = q[["Q1", "Q2", "Q3"]].apply(pd.to_timedelta).min(axis=1).dt.total_seconds()
        qq = pd.DataFrame({"Driver": q["Abbreviation"].values, "QualiPosition": q["Position"].values, "QualiBest": best.values})
        out = out.merge(qq, on="Driver", how="left")
        out["QualiGapPct"] = (out["QualiBest"] / out["QualiBest"].min() - 1) * 100
        out = out.drop(columns=["QualiBest"])
    except Exception as e:
        print("   no qualifying:", type(e).__name__)
        out["QualiPosition"], out["QualiGapPct"] = np.nan, np.nan
    out["GridPosition"] = np.where(out["GridPosition"].fillna(0) <= 0, len(out), out["GridPosition"])
    return out

have = set(zip(results["Year"], results["Round"]))
extra = []
try:
    sched = fastf1.get_event_schedule(today.year, include_testing=False)
    for _, ev in sched.iterrows():
        if pd.Timestamp(ev["EventDate"]) >= today or (today.year, int(ev["RoundNumber"])) in have:
            continue
        try:
            extra.append(fastf1_race(today.year, ev))
            print("FastF1 filled:", ev["EventName"])
        except Exception as e:
            print(f"SKIP {ev['EventName']}: {type(e).__name__}: {str(e)[:100]}")
except Exception as e:
    print("FastF1 schedule unavailable:", type(e).__name__)
if extra:
    results = pd.concat([results] + extra, ignore_index=True)

In [ ]:
cols = ["Year", "Round", "Race", "Location", "CircuitName", "EventDate", "Driver", "DriverNumber", "FullName", "TeamName", "TeamKey",
        "GridPosition", "FinalPosition", "Status", "Points", "QualiPosition", "QualiGapPct"]
spark.createDataFrame(results[cols]).write.mode("overwrite").saveAsTable(RESULTS_TABLE)

res = spark.table(RESULTS_TABLE).toPandas()
print("Rows:", len(res), "| races:", res.groupby(["Year", "Race"]).ngroups)
print(res.groupby("Year")["Race"].nunique().to_string())
print("Missing qualifying gap:", round(res["QualiGapPct"].isna().mean(), 3))
latest = res.sort_values(["EventDate", "Round"]).iloc[-1]
print("Latest race stored:", latest["Year"], latest["Race"], latest["EventDate"])